# Data Engine — Download OHLCV candles

Before we can backtest a strategy we need **price history**. This notebook uses
`CCXTLoader` to download candles from a crypto exchange and cache them locally, so
every other notebook can just *load* them instantly (no repeated downloads).

**OHLCV** = for each time bar, the **O**pen, **H**igh, **L**ow, **C**lose price
and the **V**olume traded. It is the standard input for almost every technical
strategy.

`CCXTLoader` wraps [CCXT](https://github.com/ccxt/ccxt), a library that speaks to
100+ exchanges through one unified API — so the same code works on Binance, OKX,
Bybit, etc. Downloaded candles are cached as Parquet under
`data/cache/ccxt/<exchange>/<timeframe>/<symbol>.parquet`.

In [ ]:
# CCXTLoader is our thin wrapper around CCXT that also handles local caching.
from quant_research.connectors import CCXTLoader

## Step 1 — Choose exchange, timeframe and pairs

- **exchange** — the venue id as CCXT names it. `"binanceusdm"` is Binance's
  USD-margined **perpetual futures**; `"binance"` is the spot market.
- **timeframe** — candle size: `"1m"`, `"5m"`, `"1h"`, `"1d"`, ...
- **pairs** — the symbols to fetch. `"BTC/USDT"` is spot; the extra `:USDT` in
  `"BTC/USDT:USDT"` marks a **USDT-settled perpetual** contract.

In [ ]:
exchange = "binanceusdm"  # or 'binance', 'bitget', 'okx', ...
timeframe = "1h"
pairs = [
    "BTC/USDT:USDT",   # BTC perpetual, settled in USDT
    "ETH/USDT:USDT",   # ETH perpetual
]

# One loader is bound to one exchange; it fetches that exchange's market list on init.
loader = CCXTLoader(exchange=exchange)

## Step 2 — Download and cache

`download()` pages through the exchange's history from `start_date` to now and
writes it to the local Parquet cache. This is the slow, network-bound step — but
you only pay it **once** per (exchange, timeframe, symbol). Re-running it later
only fetches the missing recent candles.

In [ ]:
for pair in pairs:
    # Fetch everything from start_date to now and store it in the cache.
    loader.download(pair, timeframe, start_date="2024-04-01 00:00:00")

## Step 3 — Load candles back

`load()` reads straight from the local cache (no network) and returns a Polars
DataFrame. Pass a `start_date`/`end_date` window to slice out just the range you
want.

In [ ]:
for pair in pairs:
    # Read a small date window back from the cache to eyeball the data.
    df = loader.load(pair, timeframe, start_date="2024-04-01 00:00:00", end_date="2024-04-05 00:00:00")
    print(pair)
    print(df)
    print()

## Step 4 — Explore what the exchange offers

`available_symbols` is every market the exchange lists. Handy when you are not
sure how a coin's pair is spelled.

In [ ]:
# Filter the full symbol list for anything containing "ETH".
looking_for = "ETH"
print([s for s in loader.available_symbols if looking_for in s][:20])

## Step 5 — Trading limits and live price

Before sizing real orders you need each market's **limits** (minimum order
amount, price precision, ...) and the current **ticker** (last/close price).
Multiplying the minimum amount by the price gives the smallest order value (in
quote currency, e.g. USDT) the exchange will accept.

In [ ]:
for symbol in pairs:
    limits = loader.market_limits(symbol)   # min/max order amount, price step, ...
    ticker = loader.ticker(symbol)          # live quote (last/close/bid/ask)
    print(f"{symbol} limits: {limits}")
    # Smallest order the exchange accepts, expressed in quote currency:
    print(f"  min notional ~ {limits['amount']['min'] * ticker['close']} quote")
    print()